In [1]:
import bw2data, bw2io
import bw2calc
import os
import re
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import colormaps
import numpy as np
import pandas as pd
import xarray as xr
#plt.rcParams['font.family'] = 'Times New Roman'
#matplotlib.style.use('tableau-colorblind10') #seaborn-v0_8-whitegrid  tableau-colorblind10

### prepare nc dataarray under SSPx, MY2020/2030/2040/2050

In [2]:
len_cf = 101  # add 1 as CFs value starting from year_0
MYs = [2030] #, 2040, 2050] #, 2060, 2070]
ssps = ["119"] #, "245", "585"]   # in raw dataarray, there is another SSP prepared "126"
sp_col, my_col = [], []

for sp in ssps: 
    repeat_sp = [sp] * len_cf * len(MYs)
    sp_col.append(repeat_sp)
sp_col_f = my_col_f = [item for sublist in sp_col for item in sublist]

for my in MYs: 
    repeat_my = [my] * len_cf  
    my_col.append(repeat_my)

my_col = my_col * len(ssps)
my_col_f = [item for sublist in my_col for item in sublist]

print(len(sp_col_f), len(my_col_f))

101 101


In [3]:
year_col =  [np.arange(len_cf)] *len(ssps) * len(MYs)
year_col_f =  [item for sublist in year_col for item in sublist]
print(len(year_col_f))

101


### extract all minorGHG names from "Fair/output/metrics" 

In [4]:
folder_path = '/Users/susierwu/dpLCA_main/dpLCIA/FaIR_dpCFs/output/metrics/minorGHG_2024-08-08'

# Define a list to store extracted substrings
all_minorghg = []

pattern = r'tstep1(.*?)_ssp'

for filename in os.listdir(folder_path):
    if filename.endswith('.xlsx') :
        # Use regex to find the substring that matches the pattern
        match = re.search(pattern, filename)

        if match:
            # If a match is found, extract the substring and add it to the list
            all_minorghg.append(match.group(1))

all_minorghg = np.unique(all_minorghg)
len(all_minorghg)

41

### read in data from FaIR output

In [5]:
cf_col_gas_allgas = []
folder_path = '/Users/susierwu/dpLCA_main/dpLCIA/FaIR_dpCFs/output/metrics/minorGHG_2024-08-08'

for ghggas in all_minorghg: 
    cf_col_gas = []
    for sp in ssps:
        for MY in MYs:
            #print(f"read point CRF value for {ghggas} SSP {sp}, model year {MY}")
            filename = "agwp_dcf_gwp100_tstep1" + ghggas + "_ssp" + sp + "_fair_start2000MY" + str(MY) + ".xlsx"
            cf_file = os.path.join(folder_path, filename)
            cf_df = pd.read_excel(cf_file, sheet_name = "agwp_pointvalue" + ghggas+ "_" + str(MY) ) 
            cf_pointvalue = cf_df[ghggas].values
            cf_col_gas.append(cf_pointvalue)
    cf_col_gas_allgas.append(cf_col_gas)

In [6]:
cf_col_gas_f = [item for sublist in cf_col_gas for item in sublist]
print(len(cf_col_gas_f))

101


In [7]:
gas_all_f_list = []
# gas_irf_all_f_list[i] is the ith gas from ghggass, flattened array for all SSPs, MYt in order 
for i in range(len(cf_col_gas_allgas)): 
    gas_all_f = [item for sublist in cf_col_gas_allgas[i] for item in sublist]
    gas_all_f_list.append(gas_all_f)

In [8]:
df_3col_names = pd.DataFrame(zip(sp_col_f, my_col_f, year_col_f), columns = ["SSP", "ModelYear", "Year"])
#2nd part of the df with CF:
col_names =    [f'CRF_{i}' for i in all_minorghg]
df_allminorghg = pd.DataFrame(np.column_stack(gas_all_f_list), columns=col_names)

cf_df = pd.concat([df_3col_names, df_allminorghg], axis=1)
cf_df

,SSP,ModelYear,Year,CRF_C2F6,CRF_C3F8,CRF_C4F10,CRF_C5F12,CRF_C6F14,CRF_C7F16,CRF_C8F18,...,CRF_HFC-365mfc,CRF_HFC-4310mee,CRF_Halon-1202,CRF_Halon-1211,CRF_Halon-1301,CRF_Halon-2402,CRF_NF3,CRF_SF6,CRF_SO2F2,CRF_c-C4F8
0,119,2030,0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
1,119,2030,1,1.058535e-11,8.034780e-12,8.668000e-12,7.918762e-12,7.430292e-12,7.254796e-12,7.125916e-12,...,8.155656e-12,7.704801e-12,6.021191e-12,9.847049e-12,1.117565e-11,6.595259e-12,1.610320e-11,2.170543e-11,1.139691e-11,8.781524e-12
2,119,2030,2,2.116964e-11,1.606648e-11,1.733267e-11,1.583559e-11,1.485819e-11,1.450717e-11,1.424946e-11,...,1.544455e-11,1.496945e-11,1.005732e-11,1.909750e-11,2.219715e-11,1.295913e-11,3.217812e-11,4.340409e-11,2.248160e-11,1.756031e-11
3,119,2030,3,3.175287e-11,2.409509e-11,2.599402e-11,2.375050e-11,2.228369e-11,2.175713e-11,2.137062e-11,...,2.195881e-11,2.181909e-11,1.276281e-11,2.778749e-11,3.306664e-11,1.909973e-11,4.822479e-11,6.509598e-11,3.326262e-11,2.633636e-11
4,119,2030,4,4.233504e-11,3.212062e-11,3.465205e-11,3.166348e-11,2.970680e-11,2.900467e-11,2.848941e-11,...,2.778074e-11,2.827744e-11,1.457636e-11,3.595098e-11,4.378620e-11,2.502489e-11,6.424328e-11,8.678112e-11,4.374828e-11,3.510968e-11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,119,2030,96,1.011382e-09,7.574450e-10,8.171391e-10,7.514727e-10,7.024946e-10,6.855446e-10,6.733661e-10,...,7.673793e-11,1.343960e-10,1.826375e-11,1.621247e-10,5.966685e-10,1.818876e-10,1.423732e-09,2.053170e-09,3.871076e-10,8.306662e-10
97,119,2030,97,1.021866e-09,7.651891e-10,8.254935e-10,7.592084e-10,7.096984e-10,6.925708e-10,6.802675e-10,...,7.673810e-11,1.344232e-10,1.826375e-11,1.621491e-10,5.996144e-10,1.821015e-10,1.437334e-09,2.074236e-09,3.878995e-10,8.391888e-10
98,119,2030,98,1.032349e-09,7.729302e-10,8.338447e-10,7.669422e-10,7.168999e-10,6.995947e-10,6.871666e-10,...,7.673825e-11,1.344488e-10,1.826375e-11,1.621721e-10,6.025196e-10,1.823079e-10,1.450912e-09,2.095295e-09,3.886697e-10,8.477088e-10
99,119,2030,99,1.042831e-09,7.806683e-10,8.421927e-10,7.746742e-10,7.240991e-10,7.066162e-10,6.940634e-10,...,7.673838e-11,1.344730e-10,1.826375e-11,1.621936e-10,6.053848e-10,1.825070e-10,1.464467e-09,2.116347e-09,3.894188e-10,8.562261e-10


In [9]:
cf_df.set_index(['SSP', 'ModelYear', 'Year'], inplace=True)

In [10]:
cf_df

CRF_C2F6      CRF_C3F8     CRF_C4F10     CRF_C5F12  \
SSP ModelYear Year                                                           
119 2030      0     0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   
              1     1.058535e-11  8.034780e-12  8.668000e-12  7.918762e-12   
              2     2.116964e-11  1.606648e-11  1.733267e-11  1.583559e-11   
              3     3.175287e-11  2.409509e-11  2.599402e-11  2.375050e-11   
              4     4.233504e-11  3.212062e-11  3.465205e-11  3.166348e-11   
...                          ...           ...           ...           ...   
              96    1.011382e-09  7.574450e-10  8.171391e-10  7.514727e-10   
              97    1.021866e-09  7.651891e-10  8.254935e-10  7.592084e-10   
              98    1.032349e-09  7.729302e-10  8.338447e-10  7.669422e-10   
              99    1.042831e-09  7.806683e-10  8.421927e-10  7.746742e-10   
              100   1.053312e-09  7.884035e-10  8.505375e-10  7.824043e-10   

                       CRF_C6F14     CRF_C7F16     CRF_C8F18      CRF_CCl4  \
SSP ModelYear Year                                                           
119 2030      0     0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   
              1     7.430292e-12  7.254796e-12  7.125916e-12  5.952723e-12   
              2     1.485819e-11  1.450717e-11  1.424946e-11  1.172230e-11   
              3     2.228369e-11  2.175713e-11  2.137062e-11  1.731437e-11   
              4     2.970680e-11  2.900467e-11  2.848941e-11  2.273438e-11   
...                          ...           ...           ...           ...   
              96    7.024946e-10  6.855446e-10  6.733661e-10  1.838463e-10   
              97    7.096984e-10  6.925708e-10  6.802675e-10  1.841426e-10   
              98    7.168999e-10  6.995947e-10  6.871666e-10  1.844299e-10   
              99    7.240991e-10  7.066162e-10  6.940634e-10  1.847083e-10   
              100   7.312959e-10  7.136354e-10  7.009579e-10  1.849781e-10   

                         CRF_CF4    CRF_CFC-11  ...  CRF_HFC-365mfc  \
SSP ModelYear Year                              ...                   
119 2030      0     0.000000e+00  0.000000e+00  ...    0.000000e+00   
              1     6.269600e-12  1.046817e-11  ...    8.155656e-12   
              2     1.253907e-11  2.073696e-11  ...    1.544455e-11   
              3     1.880842e-11  3.081016e-11  ...    2.195881e-11   
              4     2.507765e-11  4.069149e-11  ...    2.778074e-11   
...                          ...           ...  ...             ...   
              96    6.013102e-10  4.628460e-10  ...    7.673793e-11   
              97    6.075677e-10  4.644983e-10  ...    7.673810e-11   
              98    6.138252e-10  4.661192e-10  ...    7.673825e-11   
              99    6.200825e-10  4.677092e-10  ...    7.673838e-11   
              100   6.263397e-10  4.692689e-10  ...    7.673850e-11   

                    CRF_HFC-4310mee  CRF_Halon-1202  CRF_Halon-1211  \
SSP ModelYear Year                                                    
119 2030      0        0.000000e+00    0.000000e+00    0.000000e+00   
              1        7.704801e-12    6.021191e-12    9.847049e-12   
              2        1.496945e-11    1.005732e-11    1.909750e-11   
              3        2.181909e-11    1.276281e-11    2.778749e-11   
              4        2.827744e-11    1.457636e-11    3.595098e-11   
...                             ...             ...             ...   
              96       1.343960e-10    1.826375e-11    1.621247e-10   
              97       1.344232e-10    1.826375e-11    1.621491e-10   
              98       1.344488e-10    1.826375e-11    1.621721e-10   
              99       1.344730e-10    1.826375e-11    1.621936e-10   
              100      1.344957e-10    1.826375e-11    1.622138e-10   

                    CRF_Halon-1301  CRF_Halon-2402       CRF_NF3  \
SSP ModelYear Year                                                 
119 2030  

In [11]:
#Deleting rows with YEAR_index value 0
cf_df_filtered = cf_df[~cf_df.index.get_level_values("Year").isin([0])]
cf_df_filtered

CRF_C2F6      CRF_C3F8     CRF_C4F10     CRF_C5F12  \
SSP ModelYear Year                                                           
119 2030      1     1.058535e-11  8.034780e-12  8.668000e-12  7.918762e-12   
              2     2.116964e-11  1.606648e-11  1.733267e-11  1.583559e-11   
              3     3.175287e-11  2.409509e-11  2.599402e-11  2.375050e-11   
              4     4.233504e-11  3.212062e-11  3.465205e-11  3.166348e-11   
              5     5.291616e-11  4.014308e-11  4.330675e-11  3.957453e-11   
...                          ...           ...           ...           ...   
              96    1.011382e-09  7.574450e-10  8.171391e-10  7.514727e-10   
              97    1.021866e-09  7.651891e-10  8.254935e-10  7.592084e-10   
              98    1.032349e-09  7.729302e-10  8.338447e-10  7.669422e-10   
              99    1.042831e-09  7.806683e-10  8.421927e-10  7.746742e-10   
              100   1.053312e-09  7.884035e-10  8.505375e-10  7.824043e-10   

                       CRF_C6F14     CRF_C7F16     CRF_C8F18      CRF_CCl4  \
SSP ModelYear Year                                                           
119 2030      1     7.430292e-12  7.254796e-12  7.125916e-12  5.952723e-12   
              2     1.485819e-11  1.450717e-11  1.424946e-11  1.172230e-11   
              3     2.228369e-11  2.175713e-11  2.137062e-11  1.731437e-11   
              4     2.970680e-11  2.900467e-11  2.848941e-11  2.273438e-11   
              5     3.712751e-11  3.624980e-11  3.560583e-11  2.798764e-11   
...                          ...           ...           ...           ...   
              96    7.024946e-10  6.855446e-10  6.733661e-10  1.838463e-10   
              97    7.096984e-10  6.925708e-10  6.802675e-10  1.841426e-10   
              98    7.168999e-10  6.995947e-10  6.871666e-10  1.844299e-10   
              99    7.240991e-10  7.066162e-10  6.940634e-10  1.847083e-10   
              100   7.312959e-10  7.136354e-10  7.009579e-10  1.849781e-10   

                         CRF_CF4    CRF_CFC-11  ...  CRF_HFC-365mfc  \
SSP ModelYear Year                              ...                   
119 2030      1     6.269600e-12  1.046817e-11  ...    8.155656e-12   
              2     1.253907e-11  2.073696e-11  ...    1.544455e-11   
              3     1.880842e-11  3.081016e-11  ...    2.195881e-11   
              4     2.507765e-11  4.069149e-11  ...    2.778074e-11   
              5     3.134675e-11  5.038461e-11  ...    3.298394e-11   
...                          ...           ...  ...             ...   
              96    6.013102e-10  4.628460e-10  ...    7.673793e-11   
              97    6.075677e-10  4.644983e-10  ...    7.673810e-11   
              98    6.138252e-10  4.661192e-10  ...    7.673825e-11   
              99    6.200825e-10  4.677092e-10  ...    7.673838e-11   
              100   6.263397e-10  4.692689e-10  ...    7.673850e-11   

                    CRF_HFC-4310mee  CRF_Halon-1202  CRF_Halon-1211  \
SSP ModelYear Year                                                    
119 2030      1        7.704801e-12    6.021191e-12    9.847049e-12   
              2        1.496945e-11    1.005732e-11    1.909750e-11   
              3        2.181909e-11    1.276281e-11    2.778749e-11   
              4        2.827744e-11    1.457636e-11    3.595098e-11   
              5        3.436684e-11    1.579202e-11    4.361987e-11   
...                             ...             ...             ...   
              96       1.343960e-10    1.826375e-11    1.621247e-10   
              97       1.344232e-10    1.826375e-11    1.621491e-10   
              98       1.344488e-10    1.826375e-11    1.621721e-10   
              99       1.344730e-10    1.826375e-11    1.621936e-10   
              100      1.344957e-10    1.826375e-11    1.622138e-10   

                    CRF_Halon-1301  CRF_Halon-2402       CRF_NF3  \
SSP ModelYear Year                                                 
119 2030  

In [12]:
cf_xr = cf_df.to_xarray() 
cf_xr2 = cf_df_filtered.to_xarray() 

In [13]:
cf_xr.to_netcdf('data/CF_CRF0_100_perSSP_MY_minorghgs.nc')
cf_xr2.to_netcdf('data/CF_CRF1_100_perSSP_MY_minorghgs.nc')

In [14]:
cf_xr

<xarray.Dataset>
Dimensions:          (SSP: 1, ModelYear: 1, Year: 101)
Coordinates:
  * SSP              (SSP) object '119'
  * ModelYear        (ModelYear) int64 2030
  * Year             (Year) int64 0 1 2 3 4 5 6 7 8 ... 93 94 95 96 97 98 99 100
Data variables: (12/41)
    CRF_C2F6         (SSP, ModelYear, Year) float64 0.0 1.059e-11 ... 1.053e-09
    CRF_C3F8         (SSP, ModelYear, Year) float64 0.0 8.035e-12 ... 7.884e-10
    CRF_C4F10        (SSP, ModelYear, Year) float64 0.0 8.668e-12 ... 8.505e-10
    CRF_C5F12        (SSP, ModelYear, Year) float64 0.0 7.919e-12 ... 7.824e-10
    CRF_C6F14        (SSP, ModelYear, Year) float64 0.0 7.43e-12 ... 7.313e-10
    CRF_C7F16        (SSP, ModelYear, Year) float64 0.0 7.255e-12 ... 7.136e-10
    ...               ...
    CRF_Halon-1301   (SSP, ModelYear, Year) float64 0.0 1.118e-11 ... 6.082e-10
    CRF_Halon-2402   (SSP, ModelYear, Year) float64 0.0 6.595e-12 ... 1.827e-10
    CRF_NF3          (SSP, ModelYear, Year) float64 0.0 1.61e-11 ... 1.478e-09
    CRF_SF6          (SSP, ModelYear, Year) float64 0.0 2.171e-11 ... 2.137e-09
    CRF_SO2F2        (SSP, ModelYear, Year) float64 0.0 1.14e-11 ... 3.901e-10
    CRF_c-C4F8       (SSP, ModelYear, Year) float64 0.0 8.782e-12 ... 8.647e-10

### plot dCRF[1-100]